In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
import sys, os
import datetime
sys.path.append('/home/548/cd3022/repos/Irradiance-comparisons/Irradiance-comparisons')
from read_datasets import read_dataset
import logger

LOG = logger.get_logger(__name__)

from dask.distributed import Client

In [2]:
client = Client(
    n_workers=6,
    threads_per_worker=1
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 6
Total threads: 6,Total memory: 7.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46273,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:37529,Total threads: 1
Dashboard: /proxy/40251/status,Memory: 1.17 GiB
Nanny: tcp://127.0.0.1:45517,


2026-02-06 13:56:25,292 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:37529 (pid=3393754) exceeded 95% memory budget. Restarting...
2026-02-06 13:56:25,295 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:42443 (pid=3393759) exceeded 95% memory budget. Restarting...
2026-02-06 13:56:25,385 - distributed.nanny - WARNING - Restarting worker
2026-02-06 13:56:25,494 - distributed.nanny - WARNING - Restarting worker
2026-02-06 13:56:26,717 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:35303 (pid=3393783) exceeded 95% memory budget. Restarting...
2026-02-06 13:56:26,884 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:35303' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('rechunk-split-586a148fb4cc7082b83e6ccdcccd9b5f', 4), ('getitem-7e1de7f8563f571c1d8a5331fa857808', 3.1, 0, 0), ('getitem-7e1de7f8563f571c1d8a5331fa857808', 3, 0, 0)} (stimulus_id='handle-worker-cleanup-1770346586.8840

In [3]:
dataset = 'barra-r2'
year = 2019

# GHI dataset
ds_list = []
for month in range(1, 2):
    # LOAD DATASETS
    ds_month = read_dataset(
            dataset=dataset,
            resolution='hourly',
            date=f'{year}-{month:02d}'
        )
    ds_list.append(ds_month)
ds = xr.concat(ds_list, dim='time')
LOG.info(f'{dataset} data opened')

sb_path = Path('/g/data/ng72/ab4502/sea_breeze_detection/barra_c_smooth_s2/filters/')
sb_files = [f for f in sb_path.glob(f'*_F_{year}01*.zarr')]
ds_sb = xr.open_mfdataset(
    sb_files,
    engine="zarr",
    combine="by_coords",
    compat='override',
    coords='minimal',
    parallel=True,
)

2026-02-06 13:55:36,184:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)

2026-02-06 13:55:36,184:py.warnings:WARNING: /g/data/xp65/public/apps/med_conda/envs/analysis3-25.11/lib/python3.11/site-packages/xarray/backends/plugins.py:109: RuntimeWarning: Engine 'kerchunk' loading failed:
No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)

No module named 'zarr.core.array_spec'; 'zarr.core' is not a package
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)

2026-02-06 13:55:36,599:__main__:INFO: barra-r2 data opened
INFO:__main__:barra-r2 data opened
/g/data/xp65/public/apps/med_conda/envs/ana

In [6]:
# PREPROCESS DATASETS TO MATCH TIMES/SHAPES
# interp sea breeze mask from barra-c2
ds_sb = ds_sb.interp(
    lat=ds.lat,
    lon=ds.lon,
    method='nearest'
)
# ds_sb = ds_sb.persist()

# adjust hourly times on the 30min to match 3hr times (on the hour) from wf
ds_shifted = ds.assign_coords(time=ds.time - pd.Timedelta('30min'))

if dataset == 'himawari':
    # fill missing overnight time steps
    full_time = pd.date_range(
        start=ds_shifted.time.min().item(),
        end=ds_shifted.time.max().item(),
        freq="60min"
    )
    ds_shifted = ds_shifted.reindex(time=full_time, fill_value=0)
    
    # line up datasets
    min_time = ds_sb.time.min()
    max_time = ds_shifted.time.max()
    t_range = slice(min_time, max_time)
    ds_sb = ds_sb.sel(time=t_range)
    ds = ds.sel(time=t_range)

ds_times = ds_shifted.sel(time=ds_sb.time)
LOG.info('preprocessing complete')

# Extend mask to capture larger area round sea breeze
# mask = ds_sb.mask #.fillna(False).astype(bool)

sb_extension = 2 # hours
# sb_extended = (
#     mask
#     | mask.shift(time=sb_extension).fillna(False)
#     | mask.shift(time=-sb_extension).fillna(False)
# )
# sb_extended = mask.rolling(time=sb_extension*2+1).max()
# sb_extended = sb_extended.persist()

mask = (ds_sb.mask != 0)
mask_ext = mask.rolling(time=sb_extension*2+1, center=True).max().astype(bool)
mask_ext = mask_ext.persist()
LOG.info(f'sb mask extended forward and back {sb_extension} hours')

# # APPLY MASK
# sb_ghi = xr.where(sb_extended != 0, ds_times.ghi, np.nan)
# LOG.info('mask applied')

# # TIME MEAN
# sb_ghi_mean = sb_ghi.mean(dim='time')
# LOG.info('annual mean calculated')

# # remask himawari region
# if dataset == 'himawari':
#     sb_ghi_mean = xr.where(ds.isel(time=5).ghi.isnull(), np.nan, sb_ghi_mean)

ghi = ds_times.ghi
days = []
for day in ds_sb.time.dt.day:
    day = day.item()
    ghi_day = ghi.sel(time=f'{year}-01-{day:02d}')
    mask_day = mask_ext.sel(time=f'{year}-01-{day:02d}')
    
    sb_ghi_day = xr.where(mask_day != 0, ghi_day, np.nan)
    sb_ghi_day_mean = sb_ghi_day.mean(dim='time')
    sb_ghi_day_mean = sb_ghi_day_mean.assign_coords(day=day)
    days.append(sb_ghi_day_mean)
all_days = xr.concat(days, dim=day)

all_mean = all_days.mean(dim=day)
    
    
# sb_ghi = xr.where(mask != 0, ghi, np.nan)
# sb_ghi_mean = sb_ghi.mean(dim='time')

2026-02-06 13:57:10,279:__main__:INFO: preprocessing complete
INFO:__main__:preprocessing complete
2026-02-06 13:57:10,340:__main__:INFO: sb mask extended forward and back 2 hours
INFO:__main__:sb mask extended forward and back 2 hours
2026-02-06 13:57:17,324 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 0.94 GiB -- Worker memory limit: 1.17 GiB
2026-02-06 13:57:17,962 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 0.94 GiB -- Worker memory limit: 1.17 GiB
2026-02-06 13:57:24,397 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 865.87 MiB -- Worker memory limit: 1.17 GiB
2026-02-06 13:57:29,671 - distributed.worker.memory - WARNING - Worke

AttributeError: 'int' object has no attribute 'dims'

2026-02-06 13:57:30,290 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 842.00 MiB -- Worker memory limit: 1.17 GiB
2026-02-06 13:57:31,483 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 837.05 MiB -- Worker memory limit: 1.17 GiB
2026-02-06 13:57:34,782 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 0.93 GiB -- Worker memory limit: 1.17 GiB
2026-02-06 13:57:34,885 - distributed.worker.memory - WARNING - Worker is at 79% memory usage. Resuming worker. Process 

In [ ]:
all_mean.plot()

In [ ]:
# add metadata
sb_ghi_mean = sb_ghi_mean.to_dataset(name='annual_mean_ghi')
sb_ghi_mean = sb_ghi_mean.assign_coords({'year':year})
sb_ghi_mean.attrs['date_generated'] = datetime.date.today().strftime('%D')
sb_ghi_mean.attrs['source_script'] = 'data produced by the script "041.5_sea_breezes.py"'

# SAVE DATA
file_path = Path('/g/data/er8/users/cd3022/Irradiance-comparisons/weather-features/sea_breeze')
os.makedirs(file_path, exist_ok=True)
sb_ghi_mean.to_netcdf(f'{file_path}/{dataset}-{year}.nc')
LOG.info('data saved, job complete!')